In [ ]:
# ! pip install lifelines
# ! pip install Matplotlib
# ! pip install numpy
# ! pip install pandas
# ! pip install seaborn


In [ ]:
# Load libraries
import shutil
from pyspark.sql.functions import col, when
from pyspark.sql.types import StructType,StructField,DoubleType, StringType, IntegerType, DateType, TimestampType


In [ ]:
# Set config for database name, file paths, and table names
import os

database_name = os.environ.get('TELCO_DATABASE_NAME', 'telco_churn_demo')
data_path = os.environ.get('TELCO_DATA_DIR', 'data/ibm-telco-churn').rstrip('/')
driver_to_dbfs_path = '{}/Telco-Customer-Churn.csv'.format(data_path) 
bronze_tbl_path = '{}/bronze/'.format(data_path) 
silver_tbl_path = '{}/silver/'.format(data_path) 
 
bronze_tbl_name = 'bronze_customers'
silver_tbl_name = 'silver_monthly_customers'

In [ ]:
# 导入所需的库
from pyspark.sql import SparkSession

# 创建或获取 SparkSession
spark = SparkSession.builder \
    .appName("TelcoDataDeltaLake") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .getOrCreate()



In [ ]:


# Define schema
schema = StructType([
  StructField('customerID', StringType()),
  StructField('gender', StringType()),
  StructField('seniorCitizen', DoubleType()),
  StructField('partner', StringType()),
  StructField('dependents', StringType()),
  StructField('tenure', DoubleType()),
  StructField('phoneService', StringType()),
  StructField('multipleLines', StringType()),
  StructField('internetService', StringType()), 
  StructField('onlineSecurity', StringType()),
  StructField('onlineBackup', StringType()),
  StructField('deviceProtection', StringType()),
  StructField('techSupport', StringType()),
  StructField('streamingTV', StringType()),
  StructField('streamingMovies', StringType()),
  StructField('contract', StringType()),
  StructField('paperlessBilling', StringType()),
  StructField('paymentMethod', StringType()),
  StructField('monthlyCharges', DoubleType()),
  StructField('totalCharges', DoubleType()),
  StructField('churnString', StringType())
  ])

In [ ]:
print(driver_to_dbfs_path)

In [ ]:


# Load the CSV file into a DataFrame
bronze_df = spark.read.format('csv').schema(schema).option('header', 'true')\
               .load(driver_to_dbfs_path)

In [ ]:
# Construct silver table
silver_df = bronze_df.withColumn('churn',when(col('churnString') == 'Yes',1).when(col('churnString') == 'No',0).otherwise('Unknown'))\
                     .drop('churnString').filter(col('contract') == 'Month-to-month')\
                     .filter(col('internetService') != 'No')

In [ ]:
# Delete the old database and tables if needed
_ = spark.sql('DROP DATABASE IF EXISTS {} CASCADE'.format(database_name))
 
# Create database to house tables
_ = spark.sql('CREATE DATABASE {}'.format(database_name))
_ = spark.sql('USE {}'.format(database_name))

In [ ]:
# Drop any old delta lake files if needed (e.g. re-running this notebook with the same bronze_tbl_path and silver_tbl_path)
shutil.rmtree('/dbfs'+bronze_tbl_path, ignore_errors=True)
shutil.rmtree('/dbfs'+silver_tbl_path, ignore_errors=True)

In [ ]:
# Write out bronze-level data to Delta Lake
_ = bronze_df.write.mode('overwrite').save(bronze_tbl_path)

In [ ]:

# Write out silver-level data to Delta lake
_ = silver_df.write.mode('overwrite').save(silver_tbl_path)

In [ ]:

# Create bronze table
_ = spark.sql('''
  CREATE TABLE `{}`.{}
  USING parquet
  LOCATION '{}'
  '''.format(database_name,bronze_tbl_name,bronze_tbl_path))

In [ ]:
# Create silver table
_ = spark.sql('''
  CREATE TABLE `{}`.{}
  USING parquet
  LOCATION '{}'
  '''.format(database_name,silver_tbl_name,silver_tbl_path))

In [ ]:

bronze_df.show()

In [ ]:
silver_df.show()

## 01


In [ ]:
# Import libraries
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
 
from lifelines import KaplanMeierFitter
from lifelines.utils import median_survival_times
from lifelines.statistics import pairwise_logrank_test

In [ ]:

telco_pd = spark.table('silver_monthly_customers').toPandas()

In [ ]:
kmf = KaplanMeierFitter()
 
T=telco_pd['tenure']
C=telco_pd['churn'].astype(float)
 
kmf.fit(T,C)

In [ ]:
kmf.plot(title='Kaplan-Meier Survival Curve: Population level')

In [ ]:
kmf.median_survival_time_

In [ ]:
# Helper function for plotting Kaplan-Meier curves at the covariate level
def plot_km(col):
  ax = plt.subplot(111)
  for r in telco_pd[col].unique():
    ix = telco_pd[col] == r
    kmf.fit(T[ix], C[ix],label=r)
    kmf.plot(ax=ax)
    
# Helper function for printing out Log-rank test results
def print_logrank(col):
  log_rank = pairwise_logrank_test(telco_pd['tenure'], telco_pd[col], telco_pd['churn'])
  return log_rank.summary

In [ ]:
plot_km('gender')

In [ ]:
plot_km('onlineSecurity')

In [ ]:
plot_km('gender')

In [ ]:
plot_km('seniorCitizen')

In [ ]:
plot_km('partner')

In [ ]:
plot_km('dependents')

In [ ]:
plot_km('phoneService')

In [ ]:
plot_km('multipleLines')

In [ ]:
plot_km('internetService')

In [ ]:
plot_km('streamingTV')

In [ ]:
plot_km('streamingMovies')

In [ ]:
plot_km('onlineSecurity')

In [ ]:
plot_km('onlineBackup')

In [ ]:
plot_km('deviceProtection')

In [ ]:
plot_km('techSupport')

In [ ]:
plot_km('paperlessBilling')

In [ ]:
plot_km('paymentMethod')

In [ ]:
def get_survival_probs(col,val):
  ix = telco_pd[col] == val
  return kmf.fit(T[ix],C[ix],label=val)  

In [ ]:
# Extract survival probabilities for customers with internetService = DSL
sp_internet_dsl = get_survival_probs('internetService','DSL')
pd.DataFrame(sp_internet_dsl.survival_function_at_times(range(0,10)))

## 02

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(8,3))
 
ax1.plot([1,2,3,4,5],[1,2,3,4,5])
ax1.plot([1,2,3,4,5],[2,3,4,5,6])
ax1.title.set_text('Proportional')
ax1.axes.get_xaxis().set_visible(False)
ax1.axes.get_yaxis().set_visible(False)
 
ax2.plot([1,2,3,4,5],[1,2,3,4,5])
ax2.plot([1,2,3,4,5],[2,4,6,8,10])
ax2.title.set_text('Not Proportional')
ax2.axes.get_xaxis().set_visible(False)
ax2.axes.get_yaxis().set_visible(False)

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
 
from lifelines.fitters.coxph_fitter import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
from lifelines import KaplanMeierFitter

In [ ]:

telco_pd = spark.table('silver_monthly_customers').toPandas()

In [ ]:
encode_cols = ['dependents','internetService','onlineBackup','techSupport','paperlessBilling']
 
encoded_pd = pd.get_dummies(telco_pd,
               columns=encode_cols,
               prefix=encode_cols,
               drop_first=False)
 
encoded_pd.head()

In [ ]:
survival_pd = encoded_pd[['churn','tenure','dependents_Yes','internetService_DSL','onlineBackup_Yes','techSupport_Yes']]

In [ ]:

survival_pd.loc[:,'churn'] = survival_pd.loc[:,'churn'].astype('float')

In [ ]:
cph = CoxPHFitter(alpha=0.05)
cph.fit(survival_pd, 'tenure', 'churn')

In [ ]:
cph.print_summary()

In [ ]:
cph.plot(hazard_ratios=True)

In [ ]:
cph.check_assumptions(survival_pd,p_value_threshold=0.05)

In [ ]:
cph.check_assumptions(survival_pd,p_value_threshold=0.05,show_plots=True)

In [ ]:
# Fit the Kaplan-Meier model
kmf = KaplanMeierFitter()
 
T=telco_pd['tenure'] #duration
C=telco_pd['churn'].astype(float) #event observed
 
kmf.fit(T,C)

In [ ]:
# Utility function for plotting
import matplotlib.pyplot as plt
def plot_km_loglog(col):
  ax = plt.subplot(111)
  for r in telco_pd[col].unique():
    ix = telco_pd[col] == r
    kmf.fit(T[ix], C[ix],label=r)
    kmf.plot_loglogs(ax=ax)

In [ ]:
plot_km_loglog('onlineBackup')


In [ ]:
plot_km_loglog('dependents')

In [ ]:

plot_km_loglog('internetService')


In [ ]:

plot_km_loglog('techSupport')

## 04

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
 
from lifelines import WeibullAFTFitter,LogNormalAFTFitter,LogLogisticAFTFitter
from lifelines.fitters.coxph_fitter import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

In [ ]:

telco_pd = spark.table('silver_monthly_customers').toPandas()

In [ ]:
encode_cols = ['partner','multipleLines','internetService','onlineSecurity', 'onlineBackup','deviceProtection','techSupport','paymentMethod']
 
encoded_pd = pd.get_dummies(telco_pd,
               columns=encode_cols,
               prefix=encode_cols,
               drop_first=False)
 
encoded_pd.head()

In [ ]:
survival_pd = encoded_pd[['churn','tenure','partner_Yes', 'multipleLines_Yes', \
                          'internetService_DSL','onlineSecurity_Yes','onlineBackup_Yes','deviceProtection_Yes','techSupport_Yes',\
                          'paymentMethod_Bank transfer (automatic)','paymentMethod_Credit card (automatic)' ]]

In [ ]:
aft = LogLogisticAFTFitter()
aft.fit(survival_pd, duration_col='tenure', event_col='churn')

In [ ]:

print("Median Survival Time:{:.2f}".format(np.exp(aft.median_survival_time_)))

In [ ]:
aft.print_summary()

In [ ]:
aft.plot()

In [ ]:
from lifelines import KaplanMeierFitter
kmf = KaplanMeierFitter()
 
T=telco_pd['tenure'] #duration
C=telco_pd['churn'].astype(float) #event observed
 
kmf.fit(T,C)

In [ ]:
import matplotlib.pyplot as plt
 
def plot_km_logOdds(col):
  ax = plt.subplot(111)
  for r in telco_pd[col].unique():
    ix = telco_pd[col] == r
    kmf.fit(T[ix], C[ix],label=r)
    sf = kmf.survival_function_
    sf['failureOdds'] = (np.log(1-sf))/sf
    sf['logTime'] = np.log(sf.index)
    plt.plot(sf['logTime'],sf['failureOdds'])

In [ ]:
plot_km_logOdds('partner')

In [ ]:
plot_km_logOdds('multipleLines')

In [ ]:
plot_km_logOdds('internetService')

In [ ]:
plot_km_logOdds('onlineSecurity')

In [ ]:
plot_km_logOdds('onlineBackup')

In [ ]:
plot_km_logOdds('deviceProtection')

In [ ]:
plot_km_logOdds('techSupport')

In [ ]:
plot_km_logOdds('paymentMethod')

# 05

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
 
from lifelines import WeibullAFTFitter,LogNormalAFTFitter,LogLogisticAFTFitter
from lifelines.fitters.coxph_fitter import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

In [ ]:

telco_pd = spark.table('silver_monthly_customers').toPandas()

In [ ]:
encode_cols = ['dependents','internetService','onlineBackup','techSupport','paperlessBilling']
 
encoded_pd = pd.get_dummies(telco_pd,
               columns=encode_cols,
               prefix=encode_cols,
               drop_first=False)
 
encoded_pd.head()

In [ ]:

survival_pd = encoded_pd[['churn','tenure','dependents_Yes','internetService_DSL','onlineBackup_Yes','techSupport_Yes']]

In [ ]:

survival_pd.loc[:,'churn'] = survival_pd.loc[:,'churn'].astype('float')

In [ ]:
cph = CoxPHFitter(alpha=0.05)
cph.fit(survival_pd, 'tenure', 'churn')

In [ ]:
cols = ['dependents_Yes','internetService_DSL','onlineBackup_Yes','techSupport_Yes','partner_Yes',' internal rate of return']

# Mock dbutils for environments outside Databricks
class MockDBUtils:
    class Widgets:
        _widgets = {}

        @staticmethod
        def removeAll():
            print("Mock: Removed all widgets")
            MockDBUtils.Widgets._widgets.clear()

        @staticmethod
        def text(name, defaultValue):
            print(f"Mock: Created text widget '{name}' with default value '{defaultValue}'")
            MockDBUtils.Widgets._widgets[name] = defaultValue

        @staticmethod
        def dropdown(name, choices, defaultValue):
            print(f"Mock: Created dropdown widget '{name}' with choices {choices} and default value '{defaultValue}'")
            MockDBUtils.Widgets._widgets[name] = defaultValue

        @staticmethod
        def get(name):
            return MockDBUtils.Widgets._widgets.get(name, None)

    widgets = Widgets()

dbutils = MockDBUtils()

dbutils.widgets.removeAll()

for col in cols:
  if col == ' internal rate of return':
    dbutils.widgets.text(' internal rate of return', defaultValue='0.10')
  else:
    dbutils.widgets.dropdown(col, choices=['0', '1'], defaultValue='0')

In [ ]:
# Construct dataframe with values from widgets
def get_widget_values():
  widget_dict = {col:dbutils.widgets.get(col) for col in cols}
  return pd.DataFrame.from_dict(widget_dict,orient='index').T
 
def get_payback_df():
  df = get_widget_values()
  irr = df[' internal rate of return'].astype('float64')[0]/12
  cohort_df = pd.concat([pd.DataFrame([1.00]),round(cph.predict_survival_function(df),2)]).rename(columns={0:'Survival Probability'})
  cohort_df['Contract Month'] = cohort_df.index.astype('int')
  cohort_df['Monthly Profit for the Selected Plan'] = 30
  cohort_df['Avg Expected Monthly Profit'] = round(cohort_df['Survival Probability'] * cohort_df['Monthly Profit for the Selected Plan'],2)
  cohort_df['NPV of Avg Expected Monthly Profit'] = round(cohort_df['Avg Expected Monthly Profit'] / ((1+irr)**cohort_df['Contract Month']),2)
  cohort_df['Cumulative NPV'] = cohort_df['NPV of Avg Expected Monthly Profit'].cumsum()
  cohort_df['Contract Month'] = cohort_df['Contract Month'] + 1
  return cohort_df[['Contract Month','Survival Probability','Monthly Profit for the Selected Plan','Avg Expected Monthly Profit','NPV of Avg Expected Monthly Profit','Cumulative NPV']].set_index('Contract Month')


In [ ]:
pd.options.display.max_rows = 25
get_payback_df()[0:25]

In [ ]:
import seaborn as sns

# Example corrected code
ax = sns.barplot(
    x=['12 Months', '24 Months', '36 Months'],  # x-axis values
    y=get_payback_df().iloc[[11, 23, 35], :]['Cumulative NPV']  # y-axis values
)

In [ ]:
sns.lineplot(x=get_payback_df().index, y=get_payback_df()['Survival Probability'])